# Encrypted Social Media Traffic Fingerprinting under Temporal Shift A Public Benchmark and Multi-Flow Evaluation


- **Days 1–4** are development data.
- Within every Days 1–4 capture, the chronologically last **15% of flows** are reserved for validation and the preceding 85% are used for training. This keeps validation flows contiguous for legitimate multi-flow aggregation while preserving every application/capture in development.
- **Day 5 is completely untouched during model fitting, hyperparameter use, model comparison, and graph/window selection.**
- Candidate multi-flow windows are **[1, 20, 40, 60, 80] flows**.
- All five models are evaluated across all candidate windows on **validation only**.
- One **common multi-flow window** is selected using the mean validation Macro-F1 across the five models (after averaging across seeds).
- The common window is locked before Day 5.
- On **Day 5**, all five models are evaluated only at:
  1. **W = 1**, the predefined single-flow baseline; and
  2. **W = W\***, the validation-selected common multi-flow operating point.

Models: XGBoost, LightGBM, HistGradientBoosting, CatBoost, and Random Forest.

The leakage-control policy and preprocessing remain unchanged.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q lightgbm xgboost catboost shap lime

In [ ]:
import random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, log_loss
)
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

DATA_DIR = Path('/content/drive/MyDrive/mobile_merged_output')
RESULTS_DIR = Path('/content/drive/MyDrive/mobile_jcp_validation_locked_day5_results_main')
for name in ['metrics','per_class','confusion_matrices','plots','predictions']:
    (RESULTS_DIR/name).mkdir(parents=True, exist_ok=True)

SEEDS = [42,123,456,789,2026]
WINDOW_SIZES = [1,20,40,60,80]
RARE_MIN_COUNT = 10
VALIDATION_FRACTION = 0.15

## Load data

In [ ]:
files = sorted([p for p in DATA_DIR.glob('*.csv') if 'summary' not in p.name.lower()])
if not files:
    raise FileNotFoundError(f'No CSV files found in {DATA_DIR}')

frames = []
for p in files:
    d = pd.read_csv(p)
    if 'Label' not in d.columns:
        d['Label'] = p.stem.lower()
    if 'capture_file' not in d.columns:
        raise ValueError(f'{p.name} has no capture_file column')
    d['_original_row_order'] = np.arange(len(d))
    frames.append(d)
    print(f'{p.name}: {len(d):,} rows')

df = pd.concat(frames, ignore_index=True)
print('Combined:', df.shape)
display(df['Label'].value_counts().sort_index())

## Development/validation split and untouched Day-5 test

Day 5 is reserved exclusively for final testing.

For Days 1–4, validation is created **within each original capture file** after chronological ordering:

- first 85% of flows in each capture → training;
- final 15% of flows in each capture → validation;
- 100% of Day 5 → final test.

Using a contiguous validation segment is important because multi-flow validation must aggregate genuinely consecutive flows rather than a random sparse subset of a capture.

All preprocessing is fitted on training only. Candidate multi-flow window sizes are evaluated on validation only. A single common window is selected from validation and locked before Day-5 evaluation.


In [ ]:
# DEVELOPMENT / VALIDATION / TEST ASSIGNMENT
#
# Days 1–4:
#   chronologically first 85% of EACH capture -> train
#   chronologically final 15% of EACH capture -> validation
#
# Day 5:
#   100% -> untouched test
#
# This preserves genuine consecutiveness inside the validation set so that
# multi-flow window selection does not aggregate randomly scattered flows.

if 'capture_day' not in df.columns:
    raise ValueError(
        "The dataset has no 'capture_day' column. "
        "Day 5 cannot be held out without an explicit day identifier."
    )

if 'capture_file' not in df.columns:
    raise ValueError(
        "The dataset has no 'capture_file' column. "
        "Multi-flow windows must not cross capture boundaries."
    )

def normalize_day(value):
    s = str(value).strip().lower()
    digits = ''.join(ch for ch in s if ch.isdigit())
    return int(digits) if digits else np.nan

df['_day_number'] = df['capture_day'].map(normalize_day)

print('Detected capture days:')
display(df['_day_number'].value_counts(dropna=False).sort_index())

detected_days = set(df['_day_number'].dropna().astype(int).unique())

if 5 not in detected_days:
    raise ValueError(
        "Day 5 was not detected in capture_day. "
        "Inspect the printed capture_day values before continuing."
    )

unexpected_days = detected_days - {1, 2, 3, 4, 5}
if unexpected_days:
    raise ValueError(
        f'Unexpected capture days found: {sorted(unexpected_days)}. '
        'This notebook expects Days 1–5 only.'
    )

# ------------------------------------------------------------------
# Establish chronological order inside every original capture.
# ------------------------------------------------------------------
order_candidates = [
    'bidirectional_first_seen_ms',
    'src2dst_first_seen_ms',
    'dst2src_first_seen_ms',
    'id',
    '_original_row_order'
]

ORDER_COL = next(
    (c for c in order_candidates if c in df.columns),
    None
)

if ORDER_COL is None:
    raise ValueError(
        'No usable flow-order column was found.'
    )

df['_flow_order_in_capture'] = -1

for capture, idx in df.groupby('capture_file', sort=False).groups.items():

    ordered_idx = (
        df.loc[idx]
        .sort_values([ORDER_COL, '_original_row_order'])
        .index
    )

    df.loc[
        ordered_idx,
        '_flow_order_in_capture'
    ] = np.arange(len(ordered_idx))

# ------------------------------------------------------------------
# Assign splits.
# ------------------------------------------------------------------
df['_split'] = 'unused'

development_mask = df['_day_number'].isin([1, 2, 3, 4])
test_mask = df['_day_number'].eq(5)

# Day 5 is locked immediately as test.
df.loc[test_mask, '_split'] = 'test'

split_audit_rows = []

# Split EACH Days 1–4 capture chronologically.
for capture, g in df.loc[development_mask].groupby(
    'capture_file',
    sort=False
):

    g = g.sort_values(
        ['_flow_order_in_capture', '_original_row_order']
    )

    n = len(g)

    if n < 2:
        raise ValueError(
            f'Capture {capture} has fewer than 2 development flows.'
        )

    n_val = max(
        1,
        int(round(n * VALIDATION_FRACTION))
    )

    # Always retain at least one training flow.
    n_val = min(n_val, n - 1)

    train_ids = g.index[:-n_val]
    val_ids = g.index[-n_val:]

    df.loc[train_ids, '_split'] = 'train'
    df.loc[val_ids, '_split'] = 'validation'

    split_audit_rows.append({
        'capture_file': capture,
        'Label': g['Label'].iloc[0],
        'day': int(g['_day_number'].iloc[0]),
        'total_flows': n,
        'train_flows': len(train_ids),
        'validation_flows': len(val_ids),
        'validation_fraction': len(val_ids) / n
    })

if df['_split'].eq('unused').any():
    bad = df.loc[
        df['_split'].eq('unused'),
        ['capture_day', 'capture_file']
    ].drop_duplicates()

    raise RuntimeError(
        'Some rows were not assigned to a split:\n'
        + bad.to_string(index=False)
    )

split_audit = pd.DataFrame(split_audit_rows)

print('\nPer-capture development split audit:')
display(split_audit)

print('\nOverall split counts:')
display(df['_split'].value_counts())

print('\nClass counts by split:')
display(
    df.groupby(
        ['Label', '_split']
    ).size().unstack(fill_value=0)
)

print('\nDays present in each split:')
for split_name in [
    'train',
    'validation',
    'test'
]:
    days = sorted(
        df.loc[
            df['_split'].eq(split_name),
            '_day_number'
        ].dropna().unique().tolist()
    )

    print(
        f'{split_name}: {days}'
    )

# ------------------------------------------------------------------
# Hard isolation checks.
# ------------------------------------------------------------------
assert set(
    df.loc[
        df['_split'].eq('test'),
        '_day_number'
    ].unique()
) == {5}

assert 5 not in set(
    df.loc[
        df['_split'].isin(
            ['train', 'validation']
        ),
        '_day_number'
    ].unique()
)

# Each development capture must contain both train and validation.
dev_split_counts = (
    df.loc[development_mask]
    .groupby('capture_file')['_split']
    .nunique()
)

assert (
    dev_split_counts == 2
).all()

# Within each development capture, validation must begin after training.
for capture, g in df.loc[development_mask].groupby(
    'capture_file',
    sort=False
):
    tr = g.loc[
        g['_split'].eq('train'),
        '_flow_order_in_capture'
    ]
    va = g.loc[
        g['_split'].eq('validation'),
        '_flow_order_in_capture'
    ]

    assert tr.max() < va.min()

split_audit.to_csv(
    RESULTS_DIR / 'development_split_audit.csv',
    index=False
)

print(
    '\nPASS: Days 1–4 use contiguous 85/15 train-validation segments; '
    'Day 5 remains untouched.'
)


In [ ]:
# AUDIT AVAILABLE VALIDATION WINDOWS.
# These counts are used only to verify that each candidate window is feasible.
# Day 5 is not inspected for model/window selection.

val_meta_audit = df.loc[
    df['_split'].eq('validation'),
    ['capture_file', '_flow_order_in_capture']
].copy()

print(
    'Complete validation windows available '
    '(never crossing capture files):'
)

for w in WINDOW_SIZES:

    total = 0

    for _, g in val_meta_audit.groupby(
        'capture_file',
        sort=False
    ):
        total += len(g) // w

    print(
        f'{w:>2} flow(s): {total:,} validation windows'
    )


## Leakage-resistant feature preparation

In [ ]:
DROP_COLUMNS = {
    'Label','capture_day','capture_file','_day_number','_original_row_order','_split',
    '_flow_order_in_capture','id','expiration_id','application_name',
    'application_category_name','application_is_guessed','application_confidence',
    'requested_server_name','src_ip','dst_ip','src_mac','dst_mac','src_oui','dst_oui',
    'src_port','dst_port','splt_direction','splt_ps','splt_piat_ms'
}
ABSOLUTE_TIME_COLUMNS = {
    c for c in df.columns
    if c.endswith('_first_seen_ms') or c.endswith('_last_seen_ms')
}
DROP_COLUMNS |= ABSOLUTE_TIME_COLUMNS

X = df[[c for c in df.columns if c not in DROP_COLUMNS]].copy()
X = X.replace([np.inf,-np.inf], np.nan)

train_mask = df['_split'].eq('train')
train_view = X.loc[train_mask]
bad = list(train_view.columns[train_view.isna().all()])
bad += [c for c in train_view.columns if train_view[c].nunique(dropna=False) <= 1]
X = X.drop(columns=sorted(set(bad)))

le = LabelEncoder()
y = le.fit_transform(df['Label'])
class_names = le.classes_.tolist()
num_classes = len(class_names)

numeric_cols = X.select_dtypes(exclude=['object','string','category']).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print('Feature matrix:', X.shape)
print('Numeric:', len(numeric_cols))
print('Categorical:', len(categorical_cols))
print('Classes:', class_names)

## Preprocessing and models

In [ ]:
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, min_count=10, rare_token='__RARE__'):
        self.min_count = min_count
        self.rare_token = rare_token
    def fit(self, X, y=None):
        a = np.asarray(X, dtype=object)
        self.keep_ = []
        for j in range(a.shape[1]):
            s = pd.Series(a[:,j]).astype(str)
            vc = s.value_counts(dropna=False)
            self.keep_.append(set(vc[vc >= self.min_count].index))
        return self
    def transform(self, X):
        a = np.asarray(X, dtype=object).copy()
        for j, keep in enumerate(self.keep_):
            s = pd.Series(a[:,j]).astype(str)
            a[:,j] = np.where(s.isin(keep), s, self.rare_token)
        return a
    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

def make_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('rare', RareCategoryGrouper(RARE_MIN_COUNT)),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols)
    ], sparse_threshold=0)

def build_models(seed):
    return {
        'XGBoost': xgb.XGBClassifier(
            n_estimators=800, learning_rate=0.04, max_depth=6,
            min_child_weight=2, gamma=0.02, subsample=0.85,
            colsample_bytree=0.85, reg_alpha=0.05, reg_lambda=1.5,
            random_state=seed, n_jobs=-1, eval_metric='mlogloss',
            tree_method='hist', verbosity=0
        ),
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=800, learning_rate=0.04, num_leaves=31,
            max_depth=8, min_child_samples=20, subsample=0.85,
            subsample_freq=1, colsample_bytree=0.85, reg_alpha=0.05,
            reg_lambda=1.5, random_state=seed, n_jobs=-1,
            class_weight='balanced', verbose=-1
        ),
        'HistGradientBoosting': HistGradientBoostingClassifier(
            max_iter=700, learning_rate=0.05, max_leaf_nodes=31,
            max_depth=8, min_samples_leaf=20, l2_regularization=1.0,
            early_stopping=True, validation_fraction=0.15,
            n_iter_no_change=30, random_state=seed
        ),
        'CatBoost': CatBoostClassifier(
            iterations=800, learning_rate=0.05, depth=7,
            l2_leaf_reg=3.0, random_strength=0.5,
            random_seed=seed, verbose=0, allow_writing_files=False
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=600, max_depth=15, min_samples_split=8,
            min_samples_leaf=3, max_features='sqrt',
            random_state=seed, n_jobs=-1,
            class_weight='balanced_subsample'
        )
    }

## Multi-flow probability averaging

In [ ]:
def aggregate_windows(probabilities, true_labels, metadata, window_size):
    probs_out, y_out, rows = [], [], []
    m = metadata.reset_index(drop=True).copy()
    m['_p'] = np.arange(len(m))

    for capture, g in m.groupby('capture_file', sort=False):
        g = g.sort_values('_flow_order_in_capture')
        pos = g['_p'].to_numpy()

        for start in range(0, len(pos), window_size):
            idx = pos[start:start+window_size]
            if len(idx) < window_size:
                continue
            labels = np.asarray(true_labels)[idx]
            if len(np.unique(labels)) != 1:
                continue
            probs_out.append(np.asarray(probabilities)[idx].mean(axis=0))
            y_out.append(int(labels[0]))
            rows.append({
                'capture_file': capture,
                'window_size': window_size,
                'window_start_order': int(g.iloc[start]['_flow_order_in_capture']),
                'window_end_order': int(g.iloc[start+window_size-1]['_flow_order_in_capture'])
            })

    return np.asarray(probs_out), np.asarray(y_out, dtype=int), pd.DataFrame(rows)

def metric_row(y_true, probs):
    pred = probs.argmax(axis=1)
    return {
        'Accuracy': accuracy_score(y_true,pred),
        'Precision_macro': precision_score(y_true,pred,labels=np.arange(num_classes),average='macro',zero_division=0),
        'Recall_macro': recall_score(y_true,pred,labels=np.arange(num_classes),average='macro',zero_division=0),
        'F1_macro': f1_score(y_true,pred,labels=np.arange(num_classes),average='macro',zero_division=0),
        'F1_weighted': f1_score(y_true,pred,labels=np.arange(num_classes),average='weighted',zero_division=0),
        'LogLoss': log_loss(y_true,probs,labels=np.arange(num_classes)),
        'Windows': len(y_true)
    }

## Validation-based window selection, followed by locked Day-5 testing

For each seed, all five models are trained on the same training partition and evaluated on the same contiguous validation partition.

Validation evaluates every candidate window:

`W = {1, 20, 40, 60, 80}`.

After all seeds finish:

1. Macro-F1 is averaged across seeds for every `(model, window)` pair.
2. For each window, those five model-level means are averaged to obtain a **common validation score**.
3. The window with the highest common validation Macro-F1 is locked as `W*`.
4. Day 5 is then evaluated for every model only at:
   - `W = 1` (predefined single-flow baseline), and
   - `W = W*` (locked multi-flow operating point).

No Day-5 performance value is used to choose `W*`.


In [ ]:
train_idx = np.where(
    df['_split'].eq('train')
)[0]

val_idx = np.where(
    df['_split'].eq('validation')
)[0]

test_idx = np.where(
    df['_split'].eq('test')
)[0]

Xtr_raw = X.iloc[train_idx]
Xv_raw = X.iloc[val_idx]
Xte_raw = X.iloc[test_idx]

ytr = y[train_idx]
yv = y[val_idx]
yte = y[test_idx]

meta_val = (
    df.iloc[val_idx][
        [
            'capture_file',
            '_flow_order_in_capture'
        ]
    ]
    .reset_index(drop=True)
)

meta_test = (
    df.iloc[test_idx][
        [
            'capture_file',
            '_flow_order_in_capture'
        ]
    ]
    .reset_index(drop=True)
)

# ------------------------------------------------------------------
# Helper: fit each model with its appropriate validation mechanism.
# ------------------------------------------------------------------
def fit_model(
    model_name,
    model,
    Xtr,
    ytr,
    Xv,
    yv
):

    if model_name == 'XGBoost':

        model.fit(
            Xtr,
            ytr,
            eval_set=[(Xv, yv)],
            verbose=False
        )

    elif model_name == 'LightGBM':

        model.fit(
            Xtr,
            ytr,
            eval_set=[(Xv, yv)],
            callbacks=[
                lgb.early_stopping(
                    40,
                    verbose=False
                )
            ]
        )

    elif model_name == 'CatBoost':

        model.fit(
            Xtr,
            ytr,
            eval_set=(Xv, yv),
            early_stopping_rounds=40,
            verbose=False
        )

    else:

        model.fit(
            Xtr,
            ytr
        )

    return model


# ------------------------------------------------------------------
# PHASE 1 — VALIDATION WINDOW ABLATION ONLY.
# ------------------------------------------------------------------
validation_metric_rows = []

# Keep Day-5 probabilities in memory but DO NOT score or inspect them
# until the common window has been selected from validation.
test_probability_cache = {}

for seed in SEEDS:

    print(
        '\n' + '=' * 90
    )
    print(
        'SEED',
        seed
    )
    print(
        '=' * 90
    )

    np.random.seed(seed)
    random.seed(seed)

    prep = make_preprocessor()

    Xtr = prep.fit_transform(
        Xtr_raw
    )

    Xv = prep.transform(
        Xv_raw
    )

    Xte = prep.transform(
        Xte_raw
    )

    for model_name, model in build_models(
        seed
    ).items():

        print(
            'Training',
            model_name
        )

        model = fit_model(
            model_name,
            model,
            Xtr,
            ytr,
            Xv,
            yv
        )

        # Validation probabilities drive candidate-window selection.
        val_probs = model.predict_proba(
            Xv
        )

        for w in WINDOW_SIZES:

            vp, vy, vm = aggregate_windows(
                val_probs,
                yv,
                meta_val,
                w
            )

            if len(vy) == 0:
                continue

            row = {
                'Seed': seed,
                'Model': model_name,
                'Window_size': w,
                **metric_row(
                    vy,
                    vp
                )
            }

            validation_metric_rows.append(
                row
            )

            print(
                f'  validation window={w}: '
                f'acc={row["Accuracy"]:.4f}, '
                f'macroF1={row["F1_macro"]:.4f}, '
                f'windows={row["Windows"]}'
            )

        # Cache test probabilities without calculating ANY Day-5 metric.
        test_probability_cache[
            (seed, model_name)
        ] = model.predict_proba(
            Xte
        )


validation_metrics_df = pd.DataFrame(
    validation_metric_rows
)

validation_metrics_df.to_csv(
    RESULTS_DIR /
    'metrics' /
    'validation_all_seed_window_metrics.csv',
    index=False
)

# ------------------------------------------------------------------
# Select ONE COMMON WINDOW using validation Macro-F1 only.
# ------------------------------------------------------------------

# First average across seeds for each model/window.
validation_model_window = (
    validation_metrics_df
    .groupby(
        [
            'Model',
            'Window_size'
        ],
        as_index=False
    )
    .agg(
        Validation_F1_macro_mean=(
            'F1_macro',
            'mean'
        ),
        Validation_F1_macro_std=(
            'F1_macro',
            'std'
        ),
        Validation_accuracy_mean=(
            'Accuracy',
            'mean'
        ),
        Validation_windows_mean=(
            'Windows',
            'mean'
        )
    )
)

# Then average the five model means at each candidate window.
common_window_scores = (
    validation_model_window
    .groupby(
        'Window_size',
        as_index=False
    )
    .agg(
        Common_validation_macro_F1=(
            'Validation_F1_macro_mean',
            'mean'
        ),
        Common_validation_accuracy=(
            'Validation_accuracy_mean',
            'mean'
        ),
        Models_present=(
            'Model',
            'nunique'
        )
    )
)

# Require every candidate selected for comparison to have all five models.
common_window_scores = (
    common_window_scores[
        common_window_scores[
            'Models_present'
        ].eq(5)
    ]
    .sort_values(
        [
            'Common_validation_macro_F1',
            'Window_size'
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

if common_window_scores.empty:
    raise RuntimeError(
        'No candidate window had validation results for all five models.'
    )

LOCKED_WINDOW = int(
    common_window_scores.iloc[0][
        'Window_size'
    ]
)

print(
    '\n' + '=' * 100
)

print(
    'VALIDATION MODEL/WINDOW SUMMARY'
)

print(
    '=' * 100
)

display(
    validation_model_window.sort_values(
        [
            'Window_size',
            'Model'
        ]
    )
)

print(
    '\nCOMMON WINDOW SELECTION — VALIDATION ONLY'
)

display(
    common_window_scores
)

print(
    f'\nLOCKED COMMON WINDOW: '
    f'{LOCKED_WINDOW} flows'
)

validation_model_window.to_csv(
    RESULTS_DIR /
    'metrics' /
    'validation_model_window_summary.csv',
    index=False
)

common_window_scores.to_csv(
    RESULTS_DIR /
    'metrics' /
    'validation_common_window_selection.csv',
    index=False
)

with open(
    RESULTS_DIR /
    'metrics' /
    'locked_window.txt',
    'w'
) as f:
    f.write(
        str(LOCKED_WINDOW)
    )


# ------------------------------------------------------------------
# PHASE 2 — FINAL DAY-5 TEST.
#
# Day 5 is evaluated ONLY at:
#   W = 1              predefined single-flow baseline
#   W = LOCKED_WINDOW  validation-selected multi-flow point
# ------------------------------------------------------------------

FINAL_TEST_WINDOWS = sorted(
    set(
        [
            1,
            LOCKED_WINDOW
        ]
    )
)

metric_rows = []
class_rows = []
cms = {}

for seed in SEEDS:

    for model_name in build_models(
        seed
    ).keys():

        flow_probs = test_probability_cache[
            (seed, model_name)
        ]

        print(
            f'\nFINAL DAY-5 TEST: '
            f'{model_name}, seed={seed}'
        )

        for w in FINAL_TEST_WINDOWS:

            wp, wy, wm = aggregate_windows(
                flow_probs,
                yte,
                meta_test,
                w
            )

            if len(wy) == 0:
                raise RuntimeError(
                    f'No complete Day-5 windows '
                    f'for W={w}.'
                )

            metrics = metric_row(
                wy,
                wp
            )

            metric_rows.append({
                'Seed': seed,
                'Model': model_name,
                'Window_size': w,
                **metrics
            })

            pred = wp.argmax(
                axis=1
            )

            rep = classification_report(
                wy,
                pred,
                labels=np.arange(
                    num_classes
                ),
                target_names=class_names,
                output_dict=True,
                zero_division=0
            )

            for cls in class_names:

                class_rows.append({
                    'Seed': seed,
                    'Model': model_name,
                    'Window_size': w,
                    'Class': cls,
                    'Precision':
                        rep[cls]['precision'],
                    'Recall':
                        rep[cls]['recall'],
                    'F1':
                        rep[cls]['f1-score'],
                    'Support':
                        rep[cls]['support']
                })

            cms.setdefault(
                (
                    model_name,
                    w
                ),
                []
            ).append(
                confusion_matrix(
                    wy,
                    pred,
                    labels=np.arange(
                        num_classes
                    )
                )
            )

            wm['true_class'] = [
                class_names[i]
                for i in wy
            ]

            wm['predicted_class'] = [
                class_names[i]
                for i in pred
            ]

            wm['seed'] = seed
            wm['model'] = model_name

            wm.to_csv(
                RESULTS_DIR /
                'predictions' /
                (
                    f'{model_name}_'
                    f'window{w}_'
                    f'seed{seed}.csv'
                ),
                index=False
            )

            print(
                f'  W={w}: '
                f'acc={metrics["Accuracy"]:.4f}, '
                f'macroF1={metrics["F1_macro"]:.4f}, '
                f'windows={metrics["Windows"]}'
            )


metrics_df = pd.DataFrame(
    metric_rows
)

per_class_df = pd.DataFrame(
    class_rows
)

metrics_df.to_csv(
    RESULTS_DIR /
    'metrics' /
    'day5_locked_test_all_seed_metrics.csv',
    index=False
)

per_class_df.to_csv(
    RESULTS_DIR /
    'per_class' /
    'day5_locked_test_per_class_metrics.csv',
    index=False
)

# ------------------------------------------------------------------
# Select the focal model for POST-HOC XAI from FINAL DAY-5 performance.
#
# IMPORTANT:
#   - The operating window was already locked from validation.
#   - This model selection occurs only after the final Day-5 benchmark is complete.
#   - It is used only to decide which final model to explain with SHAP/LIME.
#   - It does NOT feed back into training, tuning, window selection, or test metrics.
#
# Primary criterion: mean Day-5 Macro-F1 at the locked W*.
# Tie-breakers: mean Accuracy, then model name for deterministic ordering.
# ------------------------------------------------------------------

locked_test_model_summary = (
    metrics_df[
        metrics_df['Window_size'].eq(LOCKED_WINDOW)
    ]
    .groupby('Model', as_index=False)
    .agg(
        Day5_F1_macro_mean=('F1_macro', 'mean'),
        Day5_F1_macro_std=('F1_macro', 'std'),
        Day5_accuracy_mean=('Accuracy', 'mean'),
        Day5_accuracy_std=('Accuracy', 'std')
    )
    .sort_values(
        ['Day5_F1_macro_mean', 'Day5_accuracy_mean', 'Model'],
        ascending=[False, False, True]
    )
    .reset_index(drop=True)
)

if locked_test_model_summary.empty:
    raise RuntimeError('No Day-5 results are available at the locked window.')

TEST_SELECTED_MODEL = str(
    locked_test_model_summary.iloc[0]['Model']
)

print('\nDAY-5 MODEL RANKING AT LOCKED WINDOW')
display(locked_test_model_summary)
print(
    f'Post-hoc XAI focal model: {TEST_SELECTED_MODEL} '
    f'(highest mean Day-5 Macro-F1 at W={LOCKED_WINDOW})'
)

locked_test_model_summary.to_csv(
    RESULTS_DIR / 'metrics' / 'day5_locked_model_ranking_for_xai.csv',
    index=False
)

print(
    '\nPASS: Day 5 was scored only after the common window '
    'was locked from validation.'
)


## Final Day-5 mean ± standard deviation

The following table contains **final test results only**. Each model appears at the predefined single-flow baseline (`W=1`) and the validation-selected locked multi-flow window (`W=W*`). Candidate windows that were not selected on validation are not evaluated as final Day-5 operating points.


In [ ]:
agg = metrics_df.groupby(['Model','Window_size']).agg(
    Accuracy_mean=('Accuracy','mean'), Accuracy_std=('Accuracy','std'),
    Precision_macro_mean=('Precision_macro','mean'), Precision_macro_std=('Precision_macro','std'),
    Recall_macro_mean=('Recall_macro','mean'), Recall_macro_std=('Recall_macro','std'),
    F1_macro_mean=('F1_macro','mean'), F1_macro_std=('F1_macro','std'),
    F1_weighted_mean=('F1_weighted','mean'), F1_weighted_std=('F1_weighted','std'),
    LogLoss_mean=('LogLoss','mean'), LogLoss_std=('LogLoss','std'),
    Windows_mean=('Windows','mean')
).reset_index()

for metric in ['Accuracy','Precision_macro','Recall_macro','F1_macro','F1_weighted','LogLoss']:
    agg[f'{metric}_mean_std'] = (
        agg[f'{metric}_mean'].map(lambda x:f'{x:.4f}') + ' ± ' +
        agg[f'{metric}_std'].fillna(0).map(lambda x:f'{x:.4f}')
    )

display_cols = [
    'Model','Window_size','Accuracy_mean_std','Precision_macro_mean_std',
    'Recall_macro_mean_std','F1_macro_mean_std','F1_weighted_mean_std',
    'LogLoss_mean_std','Windows_mean'
]
summary = agg[display_cols].sort_values(['Window_size','Model'])
summary.to_csv(RESULTS_DIR/'metrics'/'day5_locked_summary_mean_std.csv',index=False)
display(summary)

## Validation window-selection curves and final Day-5 comparison

The window-ablation curve is a **validation result**, not a Day-5 test result. Day 5 is shown separately only for the two locked conditions (`W=1` and `W=W*`).


In [ ]:
# ------------------------------------------------------------------
# VALIDATION CURVE — candidate-window analysis
# ------------------------------------------------------------------

for metric, ylabel, filename in [
    (
        'F1_macro',
        'Validation Macro F1',
        'validation_macro_f1_by_window_size.png'
    ),
    (
        'Accuracy',
        'Validation accuracy',
        'validation_accuracy_by_window_size.png'
    )
]:

    val_plot = (
        validation_metrics_df
        .groupby(
            [
                'Model',
                'Window_size'
            ]
        )[metric]
        .mean()
        .reset_index()
    )

    plt.figure(
        figsize=(9, 5)
    )

    for model_name, g in val_plot.groupby(
        'Model'
    ):

        g = g.sort_values(
            'Window_size'
        )

        plt.plot(
            g['Window_size'],
            g[metric],
            marker='o',
            label=model_name
        )

    plt.axvline(
        LOCKED_WINDOW,
        linestyle='--',
        linewidth=1,
        label=f'Locked W={LOCKED_WINDOW}'
    )

    plt.xlabel(
        'Multi-flow window size'
    )

    plt.ylabel(
        ylabel
    )

    plt.title(
        'Validation-Based Multi-Flow Window Selection'
    )

    plt.legend(
        fontsize=8
    )

    plt.tight_layout()

    plt.savefig(
        RESULTS_DIR /
        'plots' /
        filename,
        dpi=200
    )

    plt.show()


# ------------------------------------------------------------------
# FINAL DAY-5 COMPARISON — only W=1 and locked W*
# ------------------------------------------------------------------

day5_plot = (
    metrics_df
    .groupby(
        [
            'Model',
            'Window_size'
        ]
    )[
        [
            'Accuracy',
            'F1_macro'
        ]
    ]
    .mean()
    .reset_index()
)

for metric, ylabel, filename in [
    (
        'F1_macro',
        'Day-5 Macro F1',
        'day5_single_vs_locked_macro_f1.png'
    ),
    (
        'Accuracy',
        'Day-5 accuracy',
        'day5_single_vs_locked_accuracy.png'
    )
]:

    pivot = day5_plot.pivot(
        index='Model',
        columns='Window_size',
        values=metric
    )

    pivot.plot(
        kind='bar',
        figsize=(10, 5)
    )

    plt.ylabel(
        ylabel
    )

    plt.xlabel(
        'Model'
    )

    plt.title(
        f'Final Day-5 Test: '
        f'Single Flow vs Locked W={LOCKED_WINDOW}'
    )

    plt.xticks(
        rotation=30,
        ha='right'
    )

    plt.tight_layout()

    plt.savefig(
        RESULTS_DIR /
        'plots' /
        filename,
        dpi=200
    )

    plt.show()


## Normalized mean confusion matrices — final Day-5 conditions only

Confusion matrices are produced only for the predefined single-flow baseline and the validation-selected locked multi-flow window.


In [ ]:
for (model_name,w), matrices in cms.items():
    mean_cm = np.mean(np.stack(matrices),axis=0)
    row_sums = mean_cm.sum(axis=1,keepdims=True)
    pct = np.divide(mean_cm,row_sums,out=np.zeros_like(mean_cm,dtype=float),where=row_sums!=0)*100

    plt.figure(figsize=(11,9))
    im = plt.imshow(pct,cmap='viridis',vmin=0,vmax=100)
    plt.colorbar(im,label='Percentage (%)')
    plt.xticks(np.arange(num_classes),class_names,rotation=45,ha='right')
    plt.yticks(np.arange(num_classes),class_names)
    plt.xlabel('Predicted class')
    plt.ylabel('True class')
    plt.title(f'{model_name} - Mean Confusion Matrix (%) - {w} Flow(s)')
    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(j,i,f'{pct[i,j]:.1f}%',ha='center',va='center',fontsize=7,
                     color='white' if pct[i,j]>=50 else 'black')
    plt.tight_layout()
    safe = model_name.replace(' ','_')
    plt.savefig(RESULTS_DIR/'confusion_matrices'/f'{safe}_window{w}_percent.png',dpi=200)
    plt.show()

    pd.DataFrame(pct,index=class_names,columns=class_names).to_csv(
        RESULTS_DIR/'confusion_matrices'/f'{safe}_window{w}_percent.csv'
    )

## Per-class mean and standard deviation

In [ ]:
per_class_summary = per_class_df.groupby(
    ['Model','Window_size','Class']
).agg(
    Precision_mean=('Precision','mean'), Precision_std=('Precision','std'),
    Recall_mean=('Recall','mean'), Recall_std=('Recall','std'),
    F1_mean=('F1','mean'), F1_std=('F1','std'),
    Support_mean=('Support','mean')
).reset_index()

per_class_summary.to_csv(
    RESULTS_DIR/'per_class'/'multiflow_per_class_mean_std.csv',
    index=False
)
display(per_class_summary)

## Day-5 improvement over the predefined single-flow baseline

The improvement table compares the locked validation-selected multi-flow operating point against `W=1` on Day 5. No unselected Day-5 window is used.


In [ ]:
mean_results = (
    metrics_df
    .groupby(
        [
            'Model',
            'Window_size'
        ]
    )[
        [
            'Accuracy',
            'F1_macro'
        ]
    ]
    .mean()
    .reset_index()
)

baseline = (
    mean_results[
        mean_results.Window_size.eq(1)
    ][
        [
            'Model',
            'Accuracy',
            'F1_macro'
        ]
    ]
    .rename(
        columns={
            'Accuracy':
                'Baseline_accuracy',
            'F1_macro':
                'Baseline_macro_F1'
        }
    )
)

locked_only = (
    mean_results[
        mean_results.Window_size.eq(
            LOCKED_WINDOW
        )
    ]
    .copy()
)

improvement = locked_only.merge(
    baseline,
    on='Model',
    how='left'
)

improvement[
    'Accuracy_gain'
] = (
    improvement['Accuracy']
    - improvement[
        'Baseline_accuracy'
    ]
)

improvement[
    'Macro_F1_gain'
] = (
    improvement['F1_macro']
    - improvement[
        'Baseline_macro_F1'
    ]
)

improvement[
    'Locked_window'
] = LOCKED_WINDOW

improvement.to_csv(
    RESULTS_DIR /
    'metrics' /
    'day5_locked_improvement_over_single_flow.csv',
    index=False
)

display(
    improvement.sort_values(
        'Model'
    )
)


## Locked evaluation protocol audit

This cell verifies that the final Day-5 results contain no unselected candidate windows.


In [ ]:
assert set(
    metrics_df['Window_size'].unique()
) == set(
    FINAL_TEST_WINDOWS
)

assert set(
    FINAL_TEST_WINDOWS
) == set(
    [
        1,
        LOCKED_WINDOW
    ]
)

assert len(
    common_window_scores
) > 0

assert int(
    common_window_scores.iloc[0][
        'Window_size'
    ]
) == LOCKED_WINDOW

print(
    'PASS: final Day-5 result table contains only '
    f'W=1 and locked W={LOCKED_WINDOW}.'
)

print(
    'PASS: locked window was selected from validation '
    'before any Day-5 metric was calculated.'
)


## 13. Post-hoc explainability of the best-performing Day-5 model

The multi-flow operating window is selected and locked using **validation only**. After the final Day-5 benchmark is completed at the predefined single-flow baseline and the locked multi-flow window, the model with the highest **mean Day-5 Macro-F1 at the locked window** is selected solely as the focal model for post-hoc explainability.

SHAP and LIME are then applied to correctly classified Day-5 **individual flows** from that model. This post-hoc model choice does not alter training, preprocessing, validation-based window selection, or any reported Day-5 benchmark result.

For each application class, the section reports the top 15 contributing features from SHAP and LIME, together with SHAP-LIME agreement analysis.


In [ ]:
import shap
from lime.lime_tabular import LimeTabularExplainer

XAI_TOP_K = 15
SHAP_BACKGROUND = 200
SHAP_SAMPLES_PER_CLASS = 100
LIME_SAMPLES_PER_CLASS = 25
LIME_NUM_SAMPLES = 3000
XAI_SEED = 42

XAI_DIR = RESULTS_DIR / 'xai'
XAI_SHAP_DIR = XAI_DIR / 'shap'
XAI_LIME_DIR = XAI_DIR / 'lime'
XAI_AGREE_DIR = XAI_DIR / 'agreement'

for p in [XAI_DIR, XAI_SHAP_DIR, XAI_LIME_DIR, XAI_AGREE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('XAI output folder:', XAI_DIR)
print('Day-5 best-performing focal model:', TEST_SELECTED_MODEL)


In [ ]:
# Refit the best-performing Day-5 model deterministically for post-hoc explanation.
# Model identity is determined from mean Day-5 Macro-F1 at the already locked common window.
# This choice is explanatory only and does not feed back into benchmark selection.

selected_seed = XAI_SEED

prep_xai = make_preprocessor()

Xtr_xai = prep_xai.fit_transform(
    Xtr_raw
)

Xv_xai = prep_xai.transform(
    Xv_raw
)

Xte_xai = prep_xai.transform(
    Xte_raw
)

selected_model = build_models(
    selected_seed
)[TEST_SELECTED_MODEL]

selected_model = fit_model(
    TEST_SELECTED_MODEL,
    selected_model,
    Xtr_xai,
    ytr,
    Xv_xai,
    yv
)

feature_names = prep_xai.get_feature_names_out()

feature_names = np.asarray(
    [
        str(x)
        for x in feature_names
    ]
)

test_probs_xai = selected_model.predict_proba(
    Xte_xai
)

test_pred_xai = np.argmax(
    test_probs_xai,
    axis=1
)

print(
    'Day-5 best-performing focal model fitted:',
    TEST_SELECTED_MODEL
)

print(
    'Locked common multi-flow window:',
    LOCKED_WINDOW
)

print(
    'Transformed feature count:',
    len(feature_names)
)

print(
    'Day-5 single-flow accuracy of focal model:',
    accuracy_score(
        yte,
        test_pred_xai
    )
)


In [ ]:
# Semantic feature mapping for fair SHAP-LIME comparison.
def semantic_feature_name(encoded_feature):
    name = str(encoded_feature)

    if name.startswith('num__'):
        return name[len('num__'):]

    if name.startswith('cat__'):
        remainder = name[len('cat__'):]
        for col in sorted(categorical_cols, key=len, reverse=True):
            if remainder == col or remainder.startswith(col + '_'):
                return col
        return remainder

    return name


def clean_feature_name(encoded_feature):
    name = str(encoded_feature)
    if name.startswith('num__'):
        return name[len('num__'):]
    if name.startswith('cat__'):
        return name[len('cat__'):]
    return name


In [ ]:
# SHAP: top 15 features for each class on correctly classified Day-5 flows.

rng = np.random.default_rng(XAI_SEED)

# Background drawn from training data only.
bg_n = min(SHAP_BACKGROUND, len(Xtr_xai))
bg_idx = rng.choice(len(Xtr_xai), size=bg_n, replace=False)
X_background = Xtr_xai[bg_idx]

try:
    explainer = shap.TreeExplainer(selected_model)
except Exception as exc:
    raise RuntimeError(
        f'SHAP TreeExplainer does not support the Day-5 selected '
        f'model {TEST_SELECTED_MODEL}: {exc}. '
        'The benchmark results remain valid; only the XAI adapter would need '
        'model-specific handling.'
    )

shap_rows = []
shap_top_sets = {}
shap_semantic_sets = {}

for class_id, class_name in enumerate(class_names):
    candidates = np.where(
        (yte == class_id) &
        (test_pred_xai == class_id)
    )[0]

    if len(candidates) == 0:
        print(f'No correctly classified Day-5 flows for {class_name}; skipping SHAP.')
        shap_top_sets[class_name] = set()
        shap_semantic_sets[class_name] = set()
        continue

    take = min(SHAP_SAMPLES_PER_CLASS, len(candidates))
    chosen = rng.choice(candidates, size=take, replace=False)
    X_class = Xte_xai[chosen]

    shap_values = explainer.shap_values(X_class)

    # Handle multiclass TreeExplainer SHAP formats across SHAP versions.
    if isinstance(shap_values, list):
        class_shap = np.asarray(shap_values[class_id])
    else:
        arr = np.asarray(shap_values)
        if arr.ndim == 3:
            if arr.shape[2] == num_classes:
                class_shap = arr[:, :, class_id]
            elif arr.shape[0] == num_classes:
                class_shap = arr[class_id]
            else:
                raise ValueError(f'Unexpected SHAP array shape: {arr.shape}')
        elif arr.ndim == 2:
            class_shap = arr
        else:
            raise ValueError(f'Unexpected SHAP array shape: {arr.shape}')

    mean_abs = np.mean(np.abs(class_shap), axis=0)
    order = np.argsort(mean_abs)[::-1][:XAI_TOP_K]

    exact_set = set()
    semantic_set = set()

    for rank, feat_idx in enumerate(order, start=1):
        feat = feature_names[int(feat_idx)]
        semantic = semantic_feature_name(feat)

        exact_set.add(feat)
        semantic_set.add(semantic)

        shap_rows.append({
            'Class': class_name,
            'Rank': rank,
            'Feature': clean_feature_name(feat),
            'Encoded_feature': feat,
            'Semantic_feature': semantic,
            'Mean_abs_SHAP': float(mean_abs[int(feat_idx)]),
            'Explained_correct_flows': int(take),
        })

    shap_top_sets[class_name] = exact_set
    shap_semantic_sets[class_name] = semantic_set

shap_table = pd.DataFrame(shap_rows)
shap_table.to_csv(
    XAI_SHAP_DIR / 'day5_best_model_shap_top15_per_class.csv',
    index=False
)

for cls in class_names:
    print('\n' + '=' * 100)
    print('SHAP -', TEST_SELECTED_MODEL, '-', cls)
    display(shap_table[shap_table['Class'] == cls])


In [ ]:
# LIME: top 15 features for each class on correctly classified Day-5 flows.
#
# Zero-inclusive aggregation:
# sum(abs(local weight)) / number of explained class instances.
# A feature absent from a local explanation contributes zero.

lime_explainer = LimeTabularExplainer(
    training_data=np.asarray(Xtr_xai),
    feature_names=feature_names.tolist(),
    class_names=class_names,
    mode='classification',
    discretize_continuous=True,
    random_state=XAI_SEED
)

lime_rows = []
lime_top_sets = {}
lime_semantic_sets = {}

for class_id, class_name in enumerate(class_names):
    candidates = np.where(
        (yte == class_id) &
        (test_pred_xai == class_id)
    )[0]

    if len(candidates) == 0:
        print(f'No correctly classified Day-5 flows for {class_name}; skipping LIME.')
        lime_top_sets[class_name] = set()
        lime_semantic_sets[class_name] = set()
        continue

    take = min(LIME_SAMPLES_PER_CLASS, len(candidates))
    chosen = rng.choice(candidates, size=take, replace=False)

    sums = {}
    occurrences = {}

    for idx in chosen:
        exp = lime_explainer.explain_instance(
            np.asarray(Xte_xai[idx]),
            selected_model.predict_proba,
            labels=[class_id],
            num_features=XAI_TOP_K,
            num_samples=LIME_NUM_SAMPLES
        )

        for feat_idx, weight in exp.local_exp[class_id]:
            feat = feature_names[int(feat_idx)]
            sums[feat] = sums.get(feat, 0.0) + abs(float(weight))
            occurrences[feat] = occurrences.get(feat, 0) + 1

    averaged = {
        feat: total / take
        for feat, total in sums.items()
    }

    ranked = sorted(
        averaged.items(),
        key=lambda x: x[1],
        reverse=True
    )[:XAI_TOP_K]

    exact_set = set()
    semantic_set = set()

    for rank, (feat, importance) in enumerate(ranked, start=1):
        semantic = semantic_feature_name(feat)
        exact_set.add(feat)
        semantic_set.add(semantic)

        lime_rows.append({
            'Class': class_name,
            'Rank': rank,
            'Feature': clean_feature_name(feat),
            'Encoded_feature': feat,
            'Semantic_feature': semantic,
            'Mean_abs_LIME_zero_inclusive': float(importance),
            'Occurrence_count': int(occurrences.get(feat, 0)),
            'Explained_correct_flows': int(take),
            'Occurrence_rate': float(occurrences.get(feat, 0) / take),
        })

    lime_top_sets[class_name] = exact_set
    lime_semantic_sets[class_name] = semantic_set

lime_table = pd.DataFrame(lime_rows)
lime_table.to_csv(
    XAI_LIME_DIR / 'day5_best_model_lime_top15_per_class.csv',
    index=False
)

for cls in class_names:
    print('\n' + '=' * 100)
    print('LIME -', TEST_SELECTED_MODEL, '-', cls)
    display(lime_table[lime_table['Class'] == cls])


In [ ]:
# SHAP-LIME agreement: exact encoded features and semantic/original feature families.

agreement_rows = []

for cls in class_names:
    shap_exact = shap_top_sets.get(cls, set())
    lime_exact = lime_top_sets.get(cls, set())
    shap_sem = shap_semantic_sets.get(cls, set())
    lime_sem = lime_semantic_sets.get(cls, set())

    exact_union = shap_exact | lime_exact
    semantic_union = shap_sem | lime_sem

    exact_jaccard = (
        len(shap_exact & lime_exact) / len(exact_union)
        if exact_union else np.nan
    )
    semantic_jaccard = (
        len(shap_sem & lime_sem) / len(semantic_union)
        if semantic_union else np.nan
    )

    agreement_rows.append({
        'Class': cls,
        'Exact_Jaccard': exact_jaccard,
        'Exact_common_count': len(shap_exact & lime_exact),
        'Semantic_Jaccard': semantic_jaccard,
        'Semantic_common_count': len(shap_sem & lime_sem),
        'Exact_common_features': '; '.join(sorted(shap_exact & lime_exact)),
        'Semantic_common_features': '; '.join(sorted(shap_sem & lime_sem)),
    })

agreement_df = pd.DataFrame(agreement_rows)
agreement_df.to_csv(
    XAI_AGREE_DIR / 'day5_best_model_shap_lime_agreement_by_class.csv',
    index=False
)

display(agreement_df)

print('\nMean exact Jaccard:',
      agreement_df['Exact_Jaccard'].mean())
print('Mean semantic Jaccard:',
      agreement_df['Semantic_Jaccard'].mean())


In [ ]:
# Agreement graph.
x = np.arange(len(class_names))
width = 0.36

plt.figure(figsize=(12, 6))
plt.bar(
    x - width / 2,
    agreement_df['Exact_Jaccard'],
    width,
    label='Exact encoded features'
)
plt.bar(
    x + width / 2,
    agreement_df['Semantic_Jaccard'],
    width,
    label='Semantic feature families'
)

plt.xlabel('Application class')
plt.ylabel('Jaccard agreement')
plt.title('Best Day-5 Model SHAP-LIME Top-15 Agreement')
plt.xticks(x, class_names, rotation=45, ha='right')
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()

plot_path = XAI_AGREE_DIR / 'day5_best_model_shap_lime_top15_agreement.png'
plt.savefig(plot_path, dpi=200)
plt.show()

print('Agreement graph saved to:', plot_path)
